## **RAG DEMO - Retrival Augmented Generation**

#### 0 Setup Groq API key for LLM

In [ ]:
# Set Groq API token
# In Prod should be done through .env files
import os
os.environ['GROQ_API_KEY'] = "<YOUR API KEY HERE>"

####Setup

In [ ]:
!pip install tiktoken

In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install numpy

In [ ]:
!pip install groq

#### 1 Ingest

In [ ]:
DOCUMENT = """\
Acme Returns & Support Policy

Refunds. Opened items can be refunded within 14 days of delivery, provided they
are undamaged and you still have proof of purchase. Unopened items can be
returned within 30 days. Refunds are issued to the original payment method and
take 5 to 7 business days to appear. We do not offer cash refunds for online
orders.

Returns process. All returns require the original packaging and the original
receipt or order number. Start a return from the "My Orders" page; you will be
emailed a prepaid shipping label. We cannot accept returns of perishable goods,
gift cards, or downloadable software once the download has begun.

Shipping. Standard shipping is free on orders over $50; below that it costs
$4.99. Express shipping is a flat $14.99 and arrives in two business days.
We currently ship within India only.

Accounts. To reset your password, click "Forgot password" on the login screen
and follow the emailed link, which expires after one hour. If you are locked out
after too many attempts, your account unlocks automatically after 15 minutes.

Company. Acme was founded in 2009 and is based in Pune, India. Customer support
is available from 9am to 5pm on weekdays, via live chat and email. We do not
offer phone support.
"""
print(f"Length of Document {len(DOCUMENT)}")

Length of Document 1258


####2 Chunking

In [ ]:
# Import tiktoken and encoding "cl100k_base"

import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

In [ ]:
# Tokenize DOCUMENT

token_ids = enc.encode(DOCUMENT)
print(token_ids)

[11916, 2727, 5295, 612, 9365, 11216, 271, 4032, 44050, 13, 5377, 291, 3673, 649, 387, 94552, 2949, 220, 975, 2919, 315, 9889, 11, 3984, 814, 198, 548, 2073, 92920, 323, 499, 2103, 617, 11311, 315, 7782, 13, 1252, 57228, 3673, 649, 387, 198, 78691, 2949, 220, 966, 2919, 13, 8718, 44050, 527, 11136, 311, 279, 4113, 8323, 1749, 323, 198, 23609, 220, 20, 311, 220, 22, 2626, 2919, 311, 5101, 13, 1226, 656, 539, 3085, 8515, 73618, 369, 2930, 198, 8076, 382, 16851, 1920, 13, 2052, 4780, 1397, 279, 4113, 24066, 323, 279, 4113, 198, 63723, 477, 2015, 1396, 13, 5256, 264, 471, 505, 279, 330, 5159, 32936, 1, 2199, 26, 499, 690, 387, 198, 336, 5805, 264, 83776, 11862, 2440, 13, 1226, 4250, 4287, 4780, 315, 83217, 481, 11822, 345, 53430, 7563, 11, 477, 70752, 3241, 3131, 279, 4232, 706, 22088, 382, 44456, 13, 12028, 11862, 374, 1949, 389, 10373, 927, 400, 1135, 26, 3770, 430, 433, 7194, 198, 3, 19, 13, 1484, 13, 17855, 11862, 374, 264, 10269, 400, 975, 13, 1484, 323, 30782, 304, 1403, 2626, 2919, 

In [ ]:
# Break tokenized document into chunks using overlapping me
"""
|---------|--|------|--|--------|
0        50  60    110 120
"""

CHUNK_SIZE = 60
OVERLAP = 10
STEP = CHUNK_SIZE - OVERLAP

chunk_ids = []

for start in range(0, len(token_ids), STEP):
  chunk = token_ids[start: start+CHUNK_SIZE]
  chunk_ids.append(chunk)

for i in chunk_ids: print(f"chunk lenght {len(i)} -> {i}")


chunk lenght 60 -> [11916, 2727, 5295, 612, 9365, 11216, 271, 4032, 44050, 13, 5377, 291, 3673, 649, 387, 94552, 2949, 220, 975, 2919, 315, 9889, 11, 3984, 814, 198, 548, 2073, 92920, 323, 499, 2103, 617, 11311, 315, 7782, 13, 1252, 57228, 3673, 649, 387, 198, 78691, 2949, 220, 966, 2919, 13, 8718, 44050, 527, 11136, 311, 279, 4113, 8323, 1749, 323, 198]
chunk lenght 60 -> [44050, 527, 11136, 311, 279, 4113, 8323, 1749, 323, 198, 23609, 220, 20, 311, 220, 22, 2626, 2919, 311, 5101, 13, 1226, 656, 539, 3085, 8515, 73618, 369, 2930, 198, 8076, 382, 16851, 1920, 13, 2052, 4780, 1397, 279, 4113, 24066, 323, 279, 4113, 198, 63723, 477, 2015, 1396, 13, 5256, 264, 471, 505, 279, 330, 5159, 32936, 1, 2199]
chunk lenght 60 -> [5256, 264, 471, 505, 279, 330, 5159, 32936, 1, 2199, 26, 499, 690, 387, 198, 336, 5805, 264, 83776, 11862, 2440, 13, 1226, 4250, 4287, 4780, 315, 83217, 481, 11822, 345, 53430, 7563, 11, 477, 70752, 3241, 3131, 279, 4232, 706, 22088, 382, 44456, 13, 12028, 11862, 374, 194

In [ ]:
# Decode the ids into document strings
chunks = []

for chunk in chunk_ids:
  chunks.append(enc.decode(chunk))

print(chunks[3])

 orders over $50; below that it costs
$4.99. Express shipping is a flat $14.99 and arrives in two business days.
We currently ship within India only.

Accounts. To reset your password, click "Forgot password" on the login screen
and follow the emailed link


#### 3 Vector Embedding

In [ ]:
# Vector Encoding using Sentence Encoders using all-MiniLM-L6-v2 model (In Memory Vector Database)
# Create tuple with embeddings [("My name is Subham", [-0.4, 0.5...])]

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_vectors = model.encode(chunks, normalize_embeddings=True)



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
print(len(chunk_vectors[0]))

384


[('Acme Returns & Support Policy\n\nRefunds. Opened items can be refunded within 14 days of delivery, provided they\nare undamaged and you still have proof of purchase. Unopened items can be\nreturned within 30 days. Refunds are issued to the original payment method and\n', array([-8.10471922e-02, -2.30439492e-02,  4.51093987e-02,  9.16688740e-02,
        4.96664755e-02,  2.17373390e-02,  1.86015032e-02,  1.21556511e-02,
       -4.70489711e-02,  3.91296893e-02,  8.49436969e-02, -1.05697324e-03,
        3.86128724e-02, -5.80361523e-02, -6.21963199e-03,  7.68712256e-03,
        3.53532785e-04, -6.41112700e-02, -7.79521540e-02, -7.06435293e-02,
        1.97472945e-02,  5.22417109e-03, -4.64051105e-02, -1.15531124e-01,
        3.71663831e-04, -1.24177020e-02, -3.37495171e-02, -5.23754060e-02,
       -9.97878797e-03, -2.09026337e-02,  8.17868412e-02, -4.93871719e-02,
       -8.48726407e-02, -4.92765615e-03,  3.30288173e-03,  3.61180007e-02,
        5.09287929e-03, -1.45122439e-01, -5.965340

#### 4 Storing

In [ ]:
final_chunks = list(zip(chunks, chunk_vectors))

print(final_chunks[0:2])

[('Acme Returns & Support Policy\n\nRefunds. Opened items can be refunded within 14 days of delivery, provided they\nare undamaged and you still have proof of purchase. Unopened items can be\nreturned within 30 days. Refunds are issued to the original payment method and\n', array([-8.10471922e-02, -2.30439492e-02,  4.51093987e-02,  9.16688740e-02,
        4.96664755e-02,  2.17373390e-02,  1.86015032e-02,  1.21556511e-02,
       -4.70489711e-02,  3.91296893e-02,  8.49436969e-02, -1.05697324e-03,
        3.86128724e-02, -5.80361523e-02, -6.21963199e-03,  7.68712256e-03,
        3.53532785e-04, -6.41112700e-02, -7.79521540e-02, -7.06435293e-02,
        1.97472945e-02,  5.22417109e-03, -4.64051105e-02, -1.15531124e-01,
        3.71663831e-04, -1.24177020e-02, -3.37495171e-02, -5.23754060e-02,
       -9.97878797e-03, -2.09026337e-02,  8.17868412e-02, -4.93871719e-02,
       -8.48726407e-02, -4.92765615e-03,  3.30288173e-03,  3.61180007e-02,
        5.09287929e-03, -1.45122439e-01, -5.965340

#### 5 Retrieval - Similarity Search and Ranking

In [ ]:
# Similarity search using cosine similarity
import numpy as np

def cosine_similarity(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

In [ ]:
# Ranking and similarity search for a question
def ranked_res(question):
  # Convert the question in vector embedding
  q_vector = model.encode([question], normalize_embeddings=True)

  # Similarity Search
  ranked_chunks = []

  for s, v in final_chunks:
    score = cosine_similarity(q_vector, v)
    ranked_chunks.append((score, s))
    #print((score, s))

  # Sort
  ranked_chunks.sort(reverse=True, key=lambda x:x[0])

  # TOP 3
  context = " ".join(s for score, s in ranked_chunks[:3])

  return context


In [ ]:
question = "Where is this based out of?"
ranked_res(question)

/tmp/ipykernel_1995/2892535082.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(np.dot(a, b) / (na * nb))


' in Pune, India. Customer support\nis available from 9am to 5pm on weekdays, via live chat and email. We do not\noffer phone support.\n  on the login screen\nand follow the emailed link, which expires after one hour. If you are locked out\nafter too many attempts, your account unlocks automatically after 15 minutes.\n\nCompany. Acme was founded in 2009 and is based in Pune, India. Customer support\nis available unds are issued to the original payment method and\ntake 5 to 7 business days to appear. We do not offer cash refunds for online\norders.\n\nReturns process. All returns require the original packaging and the original\nreceipt or order number. Start a return from the "My Orders" page'

#### 6 Augment - Generating Prompt with context and Question

In [ ]:
# Question
question = "What do you think of House of Tohfa?"

In [ ]:
# Create messages/prompts
messages = [
    {"role": "system", "content": "You are a helpful assistant. Provide answer of the question from the context only. If there is no relevant answer, politely refuse with answer 'Sorry I dont have an answer for it'. Refrain adding anything about context"},
    {"role": "user", "content": f"Answer this question - {question} \n Context mentioned below \n {ranked_res(question)}"}
]

/tmp/ipykernel_1995/2892535082.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(np.dot(a, b) / (na * nb))


#### 7 Generate Answer

In [ ]:
# Provide details for Groq for Answers
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"

res = client.chat.completions.create(
    model = MODEL,
    messages = messages,
    max_tokens=500
)

print(res.choices[0].message.content)





Sorry I don't have an answer for it.
